# Mesures d'adoption — embeddings (décision D-a)
Méthode : similarité cosinus entre vecteurs, sur des paires étalonnées (cible / lié / hors-sujet, FR/EN). Modèle : `qwen3-embedding:0.6b` via Ollama (local). Prérequis : `ollama serve` + `ollama pull qwen3-embedding:0.6b`.

In [1]:
import json, math, time, urllib.request

def embed(textes):
    corps = json.dumps({"model": "qwen3-embedding:0.6b", "input": textes}).encode()
    req = urllib.request.Request("http://localhost:11434/api/embed", data=corps,
                                 headers={"Content-Type": "application/json"})
    debut = time.perf_counter()
    r = json.load(urllib.request.urlopen(req, timeout=120))
    return r["embeddings"], time.perf_counter() - debut

def cosinus(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(x * x for x in b)))

## Fondamentaux : dimension, vitesse

In [2]:
vecteurs, _ = embed(["test"])
print("dimension :", len(vecteurs[0]))
chunk = "La partie italienne commence par 1.e4 e5 2.Cf3 Cc6 3.Fc4. " * 12
_, duree = embed([chunk] * 20)
print(f"20 chunks (~400 tokens) : {duree:.2f} s → {20 / duree:.0f} chunks/s")

dimension : 1024


20 chunks (~400 tokens) : 0.41 s → 49 chunks/s


## Effet du préfixe d'instruction (sur les requêtes uniquement)

In [3]:
INSTR = "Instruct: Given a question about chess openings, retrieve relevant passages\nQuery: "
QUESTION = "Quelles sont les idées principales de la partie italienne ?"
PASSAGES = [
    ("passage italienne (cible)", "La partie italienne commence par 1.e4 e5 2.Cf3 Cc6 3.Fc4. "
     "Le fou en c4 vise le point faible f7 ; les Blancs développent vite et préparent le roque."),
    ("fou vise f7 (lié)", "Le fou en c4 vise le point faible f7 des Noirs."),
    ("gambit dame (échecs, autre sujet)", "Dans le gambit dame refusé, les Noirs soutiennent le pion d5 par e6."),
    ("tarte tatin (hors sujet)", "La tarte tatin se prépare avec des pommes caramélisées."),
    ("Italian Game EN (cible, anglais)", "In the Italian Game, White's bishop on c4 eyes the vulnerable f7 pawn."),
]
textes = [p[1] for p in PASSAGES]
sans, _ = embed([QUESTION] + textes)
avec, _ = embed([INSTR + QUESTION] + textes)
print(f"{'question ↔ passage':<38}{'sans instr':>12}{'avec instr':>12}")
scores = {}
for i, (nom, _t) in enumerate(PASSAGES):
    s, a = cosinus(sans[0], sans[i + 1]), cosinus(avec[0], avec[i + 1])
    scores[nom] = (s, a)
    print(f"{nom:<38}{s:>12.3f}{a:>12.3f}")
ecart_sans = scores["passage italienne (cible)"][0] - scores["tarte tatin (hors sujet)"][0]
ecart_avec = scores["passage italienne (cible)"][1] - scores["tarte tatin (hors sujet)"][1]
print(f"\nécart cible / hors-sujet : sans {ecart_sans:.3f} → avec {ecart_avec:.3f}")

question ↔ passage                      sans instr  avec instr
passage italienne (cible)                    0.526       0.701
fou vise f7 (lié)                            0.314       0.396
gambit dame (échecs, autre sujet)            0.250       0.396
tarte tatin (hors sujet)                     0.238       0.198
Italian Game EN (cible, anglais)             0.359       0.549

écart cible / hors-sujet : sans 0.288 → avec 0.503


## Verdict — règles adoptées
1. Embeddings via Ollama (`qwen3-embedding:0.6b`, 1024 d) : adopté.
2. Préfixe d'instruction sur les **requêtes** uniquement ; documents nus.
3. Le préfixe est un paramètre des runs A/B du gold set.

Garde-fou permanent : `backend/tests/test_embeddings_mesure.py` (sauté si Ollama absent).